# Module Analysis

Analyse module structure, inter-module dependencies, self-containment, and alignment with organic community structure from prior analysis.

In [ ]:
from __future__ import annotations
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore", category=FutureWarning)

DATA_DIR = Path("../../data/processed")
PROJECT_ROOT = Path("../..")
INTERMEDIATE_DIR = Path("../../data/intermediate")

from buildanalysis.loading import BuildDataset
from buildanalysis.modules import ModuleConfig

ds = BuildDataset(DATA_DIR, intermediate_dir=INTERMEDIATE_DIR, validate=False)

plt.rcParams.update({"figure.figsize": (12, 6), "figure.dpi": 100, "axes.titlesize": 14})
sns.set_theme(style="whitegrid", palette="muted")

## 1. Module Configuration

In [ ]:
# Load modules.yaml if available
modules_path = PROJECT_ROOT / "modules.yaml"
module_config = None
if modules_path.exists():
    module_config = ModuleConfig.from_yaml(modules_path)
    print(f"Loaded module config from {modules_path}")
    print(f"\nModule Configuration:")
    print("=" * 70)
    for mod in module_config.modules:
        patterns = ", ".join(mod.target_patterns) if mod.target_patterns else "(none)"
        dirs = ", ".join(mod.directories) if mod.directories else "(none)"
        print(f"  {mod.name:<30} ({mod.category:<20}) patterns={patterns}")
    print("=" * 70)
    print(f"Total modules: {len(module_config.modules)}")
else:
    print(f"No modules.yaml found at {modules_path}")
    print("Skipping module-specific analysis.")

In [ ]:
# Load target metrics and build graph
target_metrics = ds.target_metrics
edge_list = ds.edge_list

print(f"Target metrics: {len(target_metrics)} targets")
print(f"Edge list: {len(edge_list)} edges")

# Assign targets to modules
if module_config is not None:
    target_metrics_with_modules = module_config.assign_all_targets(target_metrics)
    assigned = target_metrics_with_modules["module"].notna().sum()
    unassigned = target_metrics_with_modules["module"].isna().sum()
    print(f"\nModule assignments: {assigned} assigned, {unassigned} unassigned")
    
    # Create module_assignments_df with standard column names (module, module_category)
    module_assignments_df = target_metrics_with_modules[["cmake_target", "module", "module_category"]].copy()
else:
    target_metrics_with_modules = target_metrics.copy()
    module_assignments_df = pd.DataFrame(columns=["cmake_target", "module", "module_category"])

# Build dependency graph using library function
from buildanalysis.graph import build_dependency_graph
bg = build_dependency_graph(target_metrics, edge_list)

import networkx as nx
G = bg.graph

print(f"\nGraph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

## 2. Module Sizes

In [ ]:
if module_config is not None:
    # Compute per-module statistics using the assigned targets
    module_stats = []
    
    for mod in module_config.modules:
        mod_targets = target_metrics_with_modules[target_metrics_with_modules["module"] == mod.name]
        
        stats = {
            "module": mod.name,
            "category": mod.category,
            "target_count": len(mod_targets),
            "total_build_time_ms": mod_targets["total_build_time_ms"].fillna(0).sum(),
            "total_sloc": mod_targets["code_lines_total"].sum() if "code_lines_total" in mod_targets.columns else 0,
        }
        module_stats.append(stats)
    
    module_stats_df = pd.DataFrame(module_stats).sort_values("total_build_time_ms", ascending=False)
    
    # Unassigned targets
    unassigned_targets = target_metrics_with_modules[target_metrics_with_modules["module"].isna()]
    print(f"\nUnassigned targets: {len(unassigned_targets)}")
    if len(unassigned_targets) > 0:
        print(f"  Examples: {', '.join(unassigned_targets['cmake_target'].head(5).tolist())}")
    
    # Plot: target count per module, build time, SLOC
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    top_by_count = module_stats_df.nlargest(15, "target_count")
    axes[0].barh(range(len(top_by_count)), top_by_count["target_count"])
    axes[0].set_yticks(range(len(top_by_count)))
    axes[0].set_yticklabels(top_by_count["module"], fontsize=9)
    axes[0].set_xlabel("Target Count")
    axes[0].set_title("Top 15 Modules by Target Count")
    axes[0].invert_yaxis()
    
    top_by_time = module_stats_df.nlargest(15, "total_build_time_ms")
    axes[1].barh(range(len(top_by_time)), top_by_time["total_build_time_ms"] / 1000)
    axes[1].set_yticks(range(len(top_by_time)))
    axes[1].set_yticklabels(top_by_time["module"], fontsize=9)
    axes[1].set_xlabel("Total Build Time (seconds)")
    axes[1].set_title("Top 15 Modules by Build Time")
    axes[1].invert_yaxis()
    
    if module_stats_df["total_sloc"].sum() > 0:
        top_by_sloc = module_stats_df[module_stats_df["total_sloc"] > 0].nlargest(15, "total_sloc")
        axes[2].barh(range(len(top_by_sloc)), top_by_sloc["total_sloc"] / 1000)
        axes[2].set_yticks(range(len(top_by_sloc)))
        axes[2].set_yticklabels(top_by_sloc["module"], fontsize=9)
        axes[2].set_xlabel("Total SLOC (thousands)")
        axes[2].set_title("Top 15 Modules by Code Volume")
        axes[2].invert_yaxis()
    
    plt.tight_layout()
    plt.show()
    
    print("\n" + "=" * 70)
    print("MODULE STATISTICS")
    print("=" * 70)
    print(module_stats_df.to_string(index=False))
else:
    print("Skipping module size analysis (no module config)")

## 3. Module Dependency Graph

In [ ]:
if module_config is not None:
    from buildanalysis.modules import build_module_dependency_graph
    
    module_graph = build_module_dependency_graph(bg, module_assignments_df)
    
    # Count inter-module edges from the raw edge list for reporting
    target_to_module = module_assignments_df.set_index("cmake_target")["module"].to_dict()
    inter_module_count = 0
    for _, row in edge_list.iterrows():
        src_mod = target_to_module.get(row["source_target"])
        dst_mod = target_to_module.get(row["dest_target"])
        if src_mod and dst_mod and src_mod != dst_mod:
            inter_module_count += 1
    
    # Count bidirectional edges
    bidirectional = 0
    for src, dst in module_graph.edges():
        if module_graph.has_edge(dst, src):
            bidirectional += 1
    bidirectional //= 2
    
    print("MODULE DEPENDENCY GRAPH")
    print("=" * 70)
    print(f"Module count: {module_graph.number_of_nodes()}")
    print(f"Inter-module edges (target-level): {inter_module_count}")
    print(f"Module-level edges: {module_graph.number_of_edges()}")
    print(f"Bidirectional edges (circular deps): {bidirectional}")
    print()
    
    if module_graph.number_of_edges() > 0:
        print("Top 20 Inter-Module Dependencies (by weight):")
        edge_data = []
        for src, dst, data in module_graph.edges(data=True):
            edge_data.append({"source_module": src, "dest_module": dst, "edge_count": data.get("weight", 1)})
        edge_counts = pd.DataFrame(edge_data).sort_values("edge_count", ascending=False)
        print(edge_counts.head(20).to_string(index=False))
else:
    module_graph = None
    print("Skipping module dependency graph analysis (no module config)")

## 4. Self-Containment

In [ ]:
if module_config is not None:
    from buildanalysis.modules import compute_module_metrics
    from buildanalysis.build import compute_critical_path
    
    # Compute critical path for enriching module metrics
    timing = target_metrics[["cmake_target", "total_build_time_ms"]].copy()
    try:
        cp = compute_critical_path(bg, timing)
        critical_path_targets = set(cp.path)
        print(f"Critical path: {len(critical_path_targets)} targets")
    except Exception:
        critical_path_targets = None
        print("Could not compute critical path (graph may have cycles)")
    
    # Use compute_module_metrics for all 19 columns
    module_metrics_df = compute_module_metrics(
        bg, module_assignments_df, target_metrics,
        critical_path_targets=critical_path_targets,
        total_targets=bg.n_targets,
    )
    
    # Display self-containment
    sc_display = module_metrics_df[["module", "category", "internal_dep_count", "external_dep_count",
                                     "self_containment", "target_count", "total_build_time_ms",
                                     "critical_path_target_count", "build_fraction"]].sort_values("self_containment")
    
    if len(sc_display) > 0:
        fig, ax = plt.subplots(figsize=(12, 8))
        sc_pct = sc_display["self_containment"] * 100
        colors = ["#d62728" if x < 50 else "#2ca02c" for x in sc_pct]
        ax.barh(range(len(sc_display)), sc_pct, color=colors)
        ax.set_yticks(range(len(sc_display)))
        ax.set_yticklabels(sc_display["module"], fontsize=9)
        ax.axvline(x=50, color="red", linestyle="--", linewidth=1, label="50% threshold")
        ax.set_xlabel("Self-Containment (%)")
        ax.set_title("Module Self-Containment")
        ax.legend()
        plt.tight_layout()
        plt.show()
    
    print("\nMODULE METRICS (all 19 columns)")
    print("=" * 120)
    print(module_metrics_df.to_string(index=False))
    
    low_containment = module_metrics_df[module_metrics_df["self_containment"] < 0.5]
    if len(low_containment) > 0:
        print(f"\nWARNING: {len(low_containment)} module(s) with <50% self-containment")
        for _, row in low_containment.iterrows():
            print(f"  - {row['module']}: {row['self_containment']*100:.0f}%")
else:
    module_metrics_df = None
    print("Skipping self-containment analysis (no module config)")

## 5. Community Alignment

In [ ]:
# Load community assignments from prior notebook (nb06)
communities_path = INTERMEDIATE_DIR / "community_assignments.parquet"
if communities_path.exists() and module_config is not None:
    communities_df = pd.read_parquet(communities_path)
    print(f"Loaded community assignments from {communities_path}")
    print(f"Communities: {communities_df['community'].nunique()}")
    
    # Rename 'target' to 'cmake_target' for merging (handle old format)
    if 'target' in communities_df.columns and 'cmake_target' not in communities_df.columns:
        communities_df = communities_df.rename(columns={"target": "cmake_target"})
    
    # Use library function for formal comparison
    from buildanalysis.modules import compare_communities_to_modules
    comparison_result = compare_communities_to_modules(communities_df, module_assignments_df)
    
    print(f"\nMODULE-COMMUNITY ALIGNMENT")
    print("=" * 80)
    print(f"  Targets compared: {comparison_result['n_targets_compared']}")
    print(f"  Adjusted Rand Index: {comparison_result['adjusted_rand_index']:.3f}")
    print(f"  Normalized Mutual Information: {comparison_result['normalized_mutual_info']:.3f}")
    if comparison_result.get('fragmented_modules'):
        print(f"  Fragmented modules: {', '.join(comparison_result['fragmented_modules'])}")
    
    # Also show per-module dominant community
    comparison_merged = communities_df.merge(module_assignments_df, on="cmake_target", how="outer")
    
    alignment = []
    for mod in module_config.modules:
        mod_targets = module_assignments_df[module_assignments_df["module"] == mod.name]["cmake_target"].tolist()
        in_comparison = comparison_merged[comparison_merged["cmake_target"].isin(mod_targets)]
        comm_counts = in_comparison["community"].dropna().value_counts()
        
        if len(comm_counts) > 0:
            dominant_comm = comm_counts.index[0]
            dominant_pct = 100 * comm_counts.iloc[0] / len(mod_targets) if len(mod_targets) > 0 else 0
            alignment.append({
                "module": mod.name,
                "dominant_community": dominant_comm,
                "dominant_pct": dominant_pct,
                "num_communities": len(comm_counts),
                "fragmented": len(comm_counts) > 1,
            })
    
    if alignment:
        alignment_df = pd.DataFrame(alignment).sort_values("dominant_pct")
        print("\nPer-Module Community Alignment:")
        print(alignment_df.to_string(index=False))
else:
    print("Skipping community alignment analysis (missing community_assignments.parquet or module config)")

## 5b. Conway's Law Alignment

Measure alignment between structural graph communities and team-based ownership, answering: does the code structure match the org structure?

In [ ]:
# Conway's Law: do structural communities match team ownership?
ownership_path = INTERMEDIATE_DIR / "target_ownership.parquet"
if communities_path.exists() and ownership_path.exists():
    from buildanalysis.modularity import compute_conway_alignment

    communities_df_conway = pd.read_parquet(communities_path)
    if "target" in communities_df_conway.columns and "cmake_target" not in communities_df_conway.columns:
        communities_df_conway = communities_df_conway.rename(columns={"target": "cmake_target"})

    ownership_conway = pd.read_parquet(ownership_path)

    # Build team-based "communities" from ownership data
    team_col = "owning_team" if "owning_team" in ownership_conway.columns else None
    target_col = "cmake_target" if "cmake_target" in ownership_conway.columns else "target"

    if team_col:
        team_communities = ownership_conway[[target_col, team_col]].dropna().copy()
        team_communities = team_communities.rename(columns={target_col: "cmake_target", team_col: "community"})

        # Encode team names as integers for ARI/NMI
        team_names = sorted(team_communities["community"].unique())
        team_to_id = {name: i for i, name in enumerate(team_names)}
        team_communities["community"] = team_communities["community"].map(team_to_id)

        conway = compute_conway_alignment(communities_df_conway, team_communities)

        print("CONWAY'S LAW ALIGNMENT")
        print("=" * 70)
        print(f"  Targets compared: {conway['n_targets_compared']}")
        print(f"  Adjusted Rand Index: {conway['adjusted_rand_index']:.3f}")
        print(f"  Normalized Mutual Information: {conway['normalized_mutual_info']:.3f}")
        if conway["adjusted_rand_index"] > 0.3:
            print("  -> Moderate-to-strong alignment: code clusters track team boundaries")
        elif conway["adjusted_rand_index"] > 0.1:
            print("  -> Weak alignment: some correspondence between structure and teams")
        else:
            print("  -> Low alignment: code structure does not follow team boundaries")
    else:
        print("Ownership data does not contain team information — skipping Conway alignment.")
else:
    print("Missing community_assignments or target_ownership — skipping Conway alignment.")

## 6. Cross-Category Dependencies

In [ ]:
if module_config is not None:
    # Build target-to-category map using standard column names
    target_to_category = {}
    for _, row in module_assignments_df.iterrows():
        if pd.notna(row["module_category"]):
            target_to_category[row["cmake_target"]] = row["module_category"]
    
    # Analyze cross-category edges
    cross_category_edges = []
    for _, row in edge_list.iterrows():
        src_cat = target_to_category.get(row["source_target"])
        dst_cat = target_to_category.get(row["dest_target"])
        
        if src_cat and dst_cat and src_cat != dst_cat:
            cross_category_edges.append({
                "source_target": row["source_target"],
                "source_category": src_cat,
                "dest_target": row["dest_target"],
                "dest_category": dst_cat,
            })
    
    if cross_category_edges:
        cross_cat_df = pd.DataFrame(cross_category_edges)
        cat_pair_counts = cross_cat_df.groupby(["source_category", "dest_category"]).size().reset_index(name="edge_count")
        cat_pair_counts = cat_pair_counts.sort_values("edge_count", ascending=False)
        
        print("\nCROSS-CATEGORY DEPENDENCIES")
        print("=" * 70)
        print(f"Total cross-category edges: {len(cross_category_edges)}")
        print("\nTop category-pair dependencies:")
        print(cat_pair_counts.to_string(index=False))
        
        categories = sorted(set(target_to_category.values()))
        if len(categories) > 1:
            cat_matrix = pd.DataFrame(0, index=categories, columns=categories)
            for _, row in cat_pair_counts.iterrows():
                cat_matrix.loc[row["source_category"], row["dest_category"]] = row["edge_count"]
            
            fig, ax = plt.subplots(figsize=(10, 8))
            sns.heatmap(cat_matrix, annot=True, fmt="d", cmap="YlOrRd", ax=ax, cbar_kws={"label": "Edge Count"})
            ax.set_title("Cross-Category Dependency Matrix")
            ax.set_xlabel("Destination Category")
            ax.set_ylabel("Source Category")
            plt.tight_layout()
            plt.show()
    else:
        print("No cross-category dependencies found.")
else:
    print("Skipping cross-category analysis (no module config)")

## 7. Save Intermediates

In [ ]:
if module_config is not None:
    ds.save_intermediate("module_assignments", module_assignments_df)
    print(f"Saved module_assignments.parquet ({len(module_assignments_df)} rows, columns: {list(module_assignments_df.columns)})")
    
    if module_metrics_df is not None and len(module_metrics_df) > 0:
        ds.save_intermediate("module_metrics", module_metrics_df)
        print(f"Saved module_metrics.parquet ({len(module_metrics_df)} rows, {len(module_metrics_df.columns)} columns)")
    
    # Save feature configurations (build set fractions per module)
    from buildanalysis.modules import build_module_feature_configs
    timing = target_metrics[["cmake_target", "total_build_time_ms"]].copy()
    feature_configs = build_module_feature_configs(bg, module_assignments_df, timing)
    ds.save_intermediate("feature_configurations", feature_configs)
    print(f"Saved feature_configurations.parquet ({len(feature_configs)} rows)")
    
    print(f"\nIntermediates saved to {INTERMEDIATE_DIR}")
else:
    print("Skipping save (no module config loaded)")